In [2]:
import openvsp as vsp
import json
import pandas as pd
import numpy as np
from collections import defaultdict

## Objective: 
Given an input wing and tail geometry file (from `surface_defn.ipynb`), this script interfaces with the OpenVSP API and models the wing, hstab, and vstab as defined. 

We further run a CompGeom analysis through the API, and access the information to calculate weights for the wing, hstab, and vstab using statistical equations from Raymer. These weights are then assigned in OpenVSP, assuming a uniform density. 

### Print Parameters
Use this function to print out all parameters associated with a body; helps troubleshooting to find the right flag to set, etc. 

In [ ]:
def print_parm_list(geom_id):
    """
    A helper function to print every parameter, its group, and its API name 
    for a given OpenVSP geometry ID.
    """
    print(f"\n--- Parameters for Geometry ID: {geom_id} ---")
    
    # get all parm IDs in this geometry's container
    parm_ids = vsp.FindContainerParmIDs(geom_id)
    
    print(f"{'GROUP NAME':<20} | {'PARAMETER NAME':<25} | {'CURRENT VALUE'}")
    print("-" * 65)
    
    # loop through and extract group, name, and value
    for p_id in parm_ids:
        group_name = vsp.GetParmGroupName(p_id)
        parm_name = vsp.GetParmName(p_id)
        
        # We try to get it as a float (double), which covers toggles and numbers
        try:
            val = vsp.GetParmVal(p_id)
        except:
            val = "N/A"
            
        print(f"{group_name:<20} | {parm_name:<25} | {val}")
    print("-" * 65 + "\n")

In [4]:
def wing_tank(wing_id, offset, eta_min, eta_max, cmin, cmax):
    wtank_id = vsp.AddGeom("CONFORMAL", wing_id)
    vsp.SetGeomName(wtank_id, "Wing Fuel Tank")

    vsp.SetParmVal(wtank_id, "Offset", "Design", offset)
    vsp.SetParmVal(wtank_id, "UTrimFlag", "Design", 1)
    vsp.SetParmVal(wtank_id, "ChordTrimFlag", "Design", 1)

    vsp.SetParmVal(wtank_id, "UMinTrimTypeFalg", "Design", 2)
    vsp.SetParmVal(wtank_id, "UMaxTrimTypeFalg", "Design", 2)

    vsp.SetParmVal(wtank_id, "ChordTrimMin", "Design", cmin)
    vsp.SetParmVal(wtank_id, "ChordTrimMax", "Design", cmax)

    vsp.SetParmVal(wtank_id, "EtaTrimMin", "Design", eta_min)
    vsp.SetParmVal(wtank_id, "EtaTrimMax", "Design", eta_max)

    print_parm_list(wtank_id)
    vsp.Update()    

    return wtank_id

### Geometry Builder

In [5]:
def auto_wing(global_x_transl, fuse, wing_foil, tail_foil, geom_def, filename):

    #Read JSON file
    with open(f"{geom_def}", "r") as file:
        geom = json.load(file)

    wing = geom["wing"]
    hstab = geom["hstab"]
    vstab = geom["vstab"]

    vsp.ClearVSPModel()


    #Insert the fuselage
    vsp.InsertVSPFile(fuse, "")


    #Define the Wing
    wing_id = vsp.AddGeom("WING", "")
    vsp.SetGeomName(wing_id, "Main_Wing")

    #Define Fold Sections
    y_fold = wing["b_fold"]
    b_half = wing["b_w"]/2
    c_root = wing["c_r_w"]
    c_tip = wing["c_t_w"]

    c_mid = c_root + ((c_tip - c_root) / b_half) * y_fold

    #Inboard Section
    vsp.InsertXSec(wing_id, 1, vsp.XS_FILE_AIRFOIL)
    vsp.Update()

    vsp.SetParmVal(wing_id, "Span", "XSec_1", y_fold)
    vsp.SetParmVal(wing_id, "Root_Chord", "XSec_1", c_root)
    vsp.SetParmVal(wing_id, "Tip_Chord", "XSec_1", c_mid)
    vsp.SetParmVal(wing_id, "Sweep_Location", "XSec_1", 0.0)
    vsp.SetParmVal(wing_id, "Sweep", "XSec_1", wing["swp_w"])
    vsp.SetParmVal(wing_id, "SectTess_U", "XSec_1", 13.0) 
    
    #Folding Outer Panel
    vsp.SetParmVal(wing_id, "Span", "XSec_2", b_half - y_fold)
    vsp.SetParmVal(wing_id, "Root_Chord", "XSec_2", c_mid)
    vsp.SetParmVal(wing_id, "Tip_Chord", "XSec_2", c_tip)
    vsp.SetParmVal(wing_id, "Sweep_Location", "XSec_2", 0.0)
    vsp.SetParmVal(wing_id, "Sweep", "XSec_2", wing["swp_w"])
    vsp.SetParmVal(wing_id, "SectTess_U", "XSec_2", 9)

    '''vsp.SetParmVal(wing_id, "Span", "XSec_1", wing["b_w"]/2)
    vsp.SetParmVal(wing_id, "Root_Chord", "XSec_1", wing["c_r_w"])
    vsp.SetParmVal(wing_id, "Tip_Chord", "XSec_1", wing["c_t_w"])
    vsp.SetParmVal(wing_id, "Sweep_Location", "XSec_1", 0)
    vsp.SetParmVal(wing_id, "Sweep", "XSec_1", wing["swp_w"])
    vsp.SetParmVal(wing_id, "SectTess_U", "XSec_1", 13.0)'''


    vsp.SetParmVal(wing_id, "X_Rel_Location", "XForm", global_x_transl)
    vsp.SetParmVal(wing_id, "Z_Rel_Location", "XForm", wing["Z_loc"])
    vsp.SetParmVal(wing_id, "Y_Rel_Rotation", "XForm", wing["Y_rot"])
    vsp.SetParmVal(wing_id, "X_Rel_Rotation", "XForm", wing["X_rot"])

    #Assign to set0
    set_0_idx = vsp.GetSetIndex("Set_0")
    set_3_idx = vsp.GetSetIndex("Set_3")
    set_19_idx = vsp.GetSetIndex("Set_19")
    vsp.SetSetFlag(wing_id, set_0_idx, True)
    vsp.SetSetFlag(wing_id, set_3_idx, True)
    vsp.SetSetFlag(wing_id, set_19_idx, True)

    #Set Wing Airfoil
    xsec_surf = vsp.GetXSecSurf(wing_id, 0)
    vsp.ChangeXSecShape(xsec_surf, 0, vsp.XS_FILE_AIRFOIL)
    vsp.ChangeXSecShape(xsec_surf, 1, vsp.XS_FILE_AIRFOIL)

    root_xsec = vsp.GetXSec(xsec_surf, 0)
    fold_xsec = vsp.GetXSec(xsec_surf, 1)
    tip_xsec = vsp.GetXSec(xsec_surf, 2)

    vsp.ReadFileAirfoil(root_xsec, wing_foil[0])
    vsp.ReadFileAirfoil(fold_xsec, wing_foil[1])
    vsp.ReadFileAirfoil(tip_xsec, wing_foil[2])

    #Fuel Tank
    #wftank_id = wing_tank(wing_id=wing_id, offset=0.05, eta_min=0.15, eta_max=y_fold/b_half - 0.02, cmin= wing["flap_c_frac1"] + 0.02, cmax= 1 - wing["slat_c_frac2"])
    wtank_id = vsp.AddGeom("CONFORMAL", wing_id)
    vsp.SetGeomName(wtank_id, "Wing Fuel Tank")

    vsp.SetParmVal(wtank_id, "Offset", "Design", 0.05)
    vsp.SetParmVal(wtank_id, "UTrimFlag", "Design", 1)
    vsp.SetParmVal(wtank_id, "ChordTrimFlag", "Design", 1)

    vsp.SetParmVal(wtank_id, "UMinTrimTypeFalg", "Design", 2)
    vsp.SetParmVal(wtank_id, "UMaxTrimTypeFalg", "Design", 2)

    vsp.SetParmVal(wtank_id, "ChordTrimMin", "Design", wing["flap_c_frac1"] + 0.02)
    vsp.SetParmVal(wtank_id, "ChordTrimMax", "Design", 1 - wing["slat_c_frac2"])

    vsp.SetParmVal(wtank_id, "EtaTrimMin", "Design", 0.15)
    vsp.SetParmVal(wtank_id, "EtaTrimMax", "Design", y_fold/b_half - 0.02)

    vsp.SetSetFlag(wtank_id, set_19_idx, True)


    #Define the Flaps
    flap_id = vsp.AddSubSurf(wing_id, vsp.SS_CONTROL)
    vsp.SetSubSurfName(wing_id, flap_id, "Flaps")
    flap_parm = vsp.GetSubSurfParmIDs(flap_id)

    for parm_id in flap_parm:
        n_parm = vsp.GetParmName(parm_id)

        if n_parm == "EtaFlag": 
            vsp.SetParmVal(parm_id, 1.0)
        #elif n_parm == "SE_Const_Flag":
        #    vsp.SetParmVal(parm_id, 0.0)
        elif n_parm == "EtaStart":
            vsp.SetParmVal(parm_id, wing["flap_1_span"])
        elif n_parm == "EtaEnd":
            vsp.SetParmVal(parm_id, wing["flap_2_span"])
        elif n_parm == "Length_C_Start":
            vsp.SetParmVal(parm_id, wing["flap_c_frac1"])
        elif n_parm == "Length_C_End":
            vsp.SetParmVal(parm_id, wing["flap_c_frac2"])

    #Define Ailerons
    ail_id = vsp.AddSubSurf(wing_id, vsp.SS_CONTROL)
    vsp.SetSubSurfName(wing_id, ail_id, "Ailerons")
    ail_parm = vsp.GetSubSurfParmIDs(ail_id)

    for parm_id in ail_parm:
        n_parm = vsp.GetParmName(parm_id)

        if n_parm == "EtaFlag": 
            vsp.SetParmVal(parm_id, 1.0)
        #elif n_parm == "SE_Const_Flag":
        #    vsp.SetParmVal(parm_id, 0.0)
        elif n_parm == "EtaStart":
            vsp.SetParmVal(parm_id, wing["ail_1_span"])
        elif n_parm == "EtaEnd":
            vsp.SetParmVal(parm_id, wing["ail_2_span"])
        elif n_parm == "Length_C_Start":
            vsp.SetParmVal(parm_id, wing["ail_c_frac1"])
        elif n_parm == "Length_C_End":
            vsp.SetParmVal(parm_id, wing["ail_c_frac2"])

    #Define Slats
    slat_id = vsp.AddSubSurf(wing_id, vsp.SS_CONTROL)
    vsp.SetSubSurfName(wing_id, slat_id, "Slats")
    slat_parm = vsp.GetSubSurfParmIDs(slat_id)

    for parm_id in slat_parm:
        n_parm = vsp.GetParmName(parm_id)

        if n_parm == "EtaFlag": 
            vsp.SetParmVal(parm_id, 1.0)
        elif n_parm == "SE_Const_Flag":
            vsp.SetParmVal(parm_id, 0.0)
        elif n_parm == "LE_Flag":
            vsp.SetParmVal(parm_id, 1.0)
        elif n_parm == "EtaStart":
            vsp.SetParmVal(parm_id, wing["slat_1_span"])
        elif n_parm == "EtaEnd":
            vsp.SetParmVal(parm_id, wing["slat_2_span"])
        elif n_parm == "Length_C_Start":
            vsp.SetParmVal(parm_id, wing["slat_c_frac1"])
        elif n_parm == "Length_C_End":
            vsp.SetParmVal(parm_id, wing["slat_c_frac2"])


    #Define the HStab
    hstab_id = vsp.AddGeom("WING", "")
    vsp.SetGeomName(hstab_id, "HStab")
    vsp.SetSetFlag(hstab_id, set_0_idx, True)
    vsp.SetSetFlag(hstab_id, set_3_idx, True)
    vsp.SetSetFlag(hstab_id, set_19_idx, True)

    vsp.SetParmVal(hstab_id, "Span", "XSec_1", hstab["b_HT"]/2)
    vsp.SetParmVal(hstab_id, "Root_Chord", "XSec_1", hstab["c_r_HT"])
    vsp.SetParmVal(hstab_id, "Tip_Chord", "XSec_1", hstab["c_t_HT"])
    vsp.SetParmVal(hstab_id, "Sweep_Location", "XSec_1", 0)
    vsp.SetParmVal(hstab_id, "Sweep", "XSec_1", hstab["swp_HT"])
    vsp.SetParmVal(hstab_id, "SectTess_U", "XSec_1", 13)
    vsp.SetParmVal(hstab_id, "X_Rel_Location", "XForm", hstab["x_loc_HT"] + global_x_transl)
    vsp.SetParmVal(hstab_id, "Y_Rel_Location", "XForm", hstab["Y_loc"])
    vsp.SetParmVal(hstab_id, "Z_Rel_Location", "XForm", hstab["Z_loc"])

    #Define the HStab Airfoil
    hstab_xsec_surf = vsp.GetXSecSurf(hstab_id, 0)
    vsp.ChangeXSecShape(hstab_xsec_surf, 0, vsp.XS_FILE_AIRFOIL)
    vsp.ChangeXSecShape(hstab_xsec_surf, 1, vsp.XS_FILE_AIRFOIL)

    hstab_root_xsec = vsp.GetXSec(hstab_xsec_surf, 0)
    hstab_tip_xsec = vsp.GetXSec(hstab_xsec_surf, 1)

    vsp.ReadFileAirfoil(hstab_root_xsec, tail_foil[1])
    vsp.ReadFileAirfoil(hstab_tip_xsec, tail_foil[0])


    #Define the Vstab
    vstab_id = vsp.AddGeom("WING", "")
    vsp.SetGeomName(vstab_id, "VStab")
    vsp.SetSetFlag(vstab_id, set_0_idx, True)
    vsp.SetSetFlag(vstab_id, set_3_idx, True)
    vsp.SetSetFlag(vstab_id, set_19_idx, True)

    vsp.SetParmVal(vstab_id, "Span", "XSec_1", vstab["b_VT"]/2)
    vsp.SetParmVal(vstab_id, "Root_Chord", "XSec_1", vstab["c_r_VT"])
    vsp.SetParmVal(vstab_id, "Tip_Chord", "XSec_1", vstab["c_t_VT"])
    vsp.SetParmVal(vstab_id, "Sweep_Location", "XSec_1", 0)
    vsp.SetParmVal(vstab_id, "Sweep", "XSec_1", vstab["swp_VT"])
    vsp.SetParmVal(vstab_id, "SectTess_U", "XSec_1", 11.0)
    vsp.SetParmVal(vstab_id, "X_Rel_Location", "XForm", vstab["x_loc_VT"] + global_x_transl)
    vsp.SetParmVal(vstab_id, "Y_Rel_Location", "XForm", vstab["Y_loc"])
    vsp.SetParmVal(vstab_id, "Z_Rel_Location", "XForm", vstab["Z_loc"])
    vsp.SetParmVal(vstab_id, "X_Rel_Rotation", "XForm", vstab["X_rot"])
    vsp.SetParmVal(vstab_id, "CapBoundFlag", "Endcap", 0.0)
    vsp.SetParmVal(vstab_id, "WakeRootFlag", "Endcap", 0.0)

    #Define the VStab Airfoil
    vstab_xsec_surf = vsp.GetXSecSurf(vstab_id, 0)
    vsp.ChangeXSecShape(vstab_xsec_surf, 0, vsp.XS_FILE_AIRFOIL)
    vsp.ChangeXSecShape(vstab_xsec_surf, 1, vsp.XS_FILE_AIRFOIL)

    vstab_root_xsec = vsp.GetXSec(vstab_xsec_surf, 0)
    vstab_tip_xsec = vsp.GetXSec(vstab_xsec_surf, 1)

    vsp.ReadFileAirfoil(vstab_root_xsec, tail_foil[0])
    vsp.ReadFileAirfoil(vstab_tip_xsec, tail_foil[0])

    #Define the Rudder
    rudder_id = vsp.AddSubSurf(vstab_id, vsp.SS_CONTROL)
    vsp.SetSubSurfName(vstab_id, rudder_id, "Rudder")
    rud_parm = vsp.GetSubSurfParmIDs(rudder_id)

    for parm_id in rud_parm:
        n_parm = vsp.GetParmName(parm_id)

        if n_parm == "EtaFlag":
            vsp.SetParmVal(parm_id, 1.0)
        elif n_parm == "EtaStart":
            vsp.SetParmVal(parm_id, vstab["rud_1_span"])
        elif n_parm == "EtaEnd":
            vsp.SetParmVal(parm_id, vstab["rud_2_span"])
        elif n_parm == "Length_C_Start":
            vsp.SetParmVal(parm_id, vstab["rud_c_frac"])
        elif n_parm == "Length_C_End":
            vsp.SetParmVal(parm_id, vstab["rud_c_frac"])


    vsp.Update()
    vsp.WriteVSPFile(filename)


### MAC FILES ###
'''wing_foil = "/Users/ryuyaiwase/Desktop/Airfoil Libby/NASA SC(2)-0406.dat"
tail_foil = "/Users/ryuyaiwase/Desktop/Airfoil Libby/NACA_16-006.dat"
fuselage = "/Users/ryuyaiwase/Desktop/OpenVSP-3.47.0-MacOS/VSP Files/SIMPLE_F24HH_FUSE.vsp3"
'''
### WINDOWS FILES ###
#wing_foil = r"C:\Users\14153\Desktop\Airfoil Library\NASA SC(2)-0404 AIRFOIL.dat"
wing_foil = [r"C:\Users\14153\Desktop\Airfoil Library\NACA 64A006_TEST.dat", r"C:\Users\14153\Desktop\Airfoil Library\NACA 64A005.dat", r"C:\Users\14153\Desktop\Airfoil Library\NACA 64A004_TEST.dat"] #Root, Mid, Tip
#tail_foil = r"C:\Users\14153\Desktop\Airfoil Library\NACA 16-006.dat"
tail_foil = [r"C:\Users\14153\Desktop\Airfoil Library\NACA 65A004.dat", r"C:\Users\14153\Desktop\Airfoil Library\NACA 65A005.dat"] #Vstab total & Hstab tip, HStab Root
fuselage = r"C:\Users\14153\Desktop\OpenVSP-3.48.2-win64\VSPFiles\2_SIMPLE_F24HH_FUSE.vsp3"

vsp_filename = "F24HH"

auto_wing(global_x_transl=16, fuse=fuselage, wing_foil=wing_foil, tail_foil=tail_foil, geom_def="airplane_geom.json", filename=f"{vsp_filename}.vsp3")

### Run CompGeom
Runs compgeom, returns three lists: volumes of each component, wetted areas of components, (wetted) areas of control surfaces.

In [6]:
def compgeom(plane):
    vsp.ClearVSPModel()

    #Insert previously defined aircraft
    vsp.ReadVSPFile(plane)

    #Run CompGeom
    print(f"Running CompGeom on {plane}...")
    vsp.ComputeCompGeom(vsp.SET_ALL, False, 1) # Set last to 1 to get text printout, 0 for none

    #Access Results
    compgeom_res_id = vsp.FindLatestResultsID("Comp_Geom")

    #Debugging, print all compgeom data names
    '''dat_names = vsp.GetAllDataNames(compgeom_res_id) 
    display(dat_names)'''

    #Theoretical volumes, and corresponding component names
    part_names = vsp.GetStringResults(compgeom_res_id, "Comp_Name")
    theo_vols = vsp.GetDoubleResults(compgeom_res_id, "Theo_Vol")

    comp_vol = defaultdict(float)
    for name, vol in zip(part_names, theo_vols):
        # Clean up the name string (removes trailing spaces from VSP)
        clean_name = name.strip() 
        comp_vol[clean_name] += vol

    print("Component Volumes [ft^3]")
    display(comp_vol)

    #Theoretica (wetted) Areas of components
    wet_areas = vsp.GetDoubleResults(compgeom_res_id, "Theo_Area")
    
    comp_wet_area = defaultdict(float)
    for name, area in zip(part_names, wet_areas):
        clean_name = name.strip()
        comp_wet_area[clean_name] += area

    print("Component Areas [ft^2]")
    display(comp_wet_area)

    #Theoretical Surface Areas, Control Surfaces ## NOTICE: OPENVSP REPORTS THEORETICAL AREA, MEANING WETTED AREA!! Divide by 2 to get planform area.
    ss_names = vsp.GetStringResults(compgeom_res_id, "SubSurf_Name")
    ss_wet_areas = vsp.GetDoubleResults(compgeom_res_id, "SubSurf_Theo_Area")
    ss_plan_areas = [x / 2 for x in ss_wet_areas]

    ss_areas = dict(zip(ss_names, ss_plan_areas))

    print("Control Surface Areas [ft^2]")
    display(ss_areas)

    vsp.Update()
    #vsp.WriteVSPFile(f"{vsp_filename}.vsp3")

    return comp_vol, comp_wet_area, ss_areas

plane = f"{vsp_filename}.vsp3"
comp_vols, comp_area, subsurf_areas = compgeom(plane=plane)

Running CompGeom on F24HH.vsp3...
Component Volumes [ft^3]


defaultdict(float,
            {'Fuselage': 533.7670323327628,
             'Canopy': 56.90622640668182,
             'Seat': 5.170283742222248,
             'Pilot (Female 20%)': 3.081204126177726,
             'Air Intake': 109.47659559842836,
             'S Ducts': -179.22738003309036,
             'Main Fuel Tank': 161.6884698594373,
             'Main Fuel Tank 2': 12.000000000000036,
             'Main Fuel Tank 3': 9.000000000000014,
             'ANAPG-81 Radar': 2.9375842152887404,
             'Avioincs': 4.8750000000000036,
             'F100-PW-229 (Engine)': -3.253273873611725e-15,
             'External Casing': 132.0953539389112,
             'Nozzle Exit': 93.12284105023976,
             'Front Tire': 0.7973477725364067,
             'Front Struct': 0.2500308415854371,
             'Diagonal Struct': 0.2500179565263997,
             'Front Tire (Stored)': 0.7973477725362913,
             'Front Struct (Stored)': 0.25001830895702903,
             'Diagonal Struct (Store

Component Areas [ft^2]


defaultdict(float,
            {'Fuselage': 653.1864380045045,
             'Canopy': 107.84982141373096,
             'Seat': 26.433664376707885,
             'Pilot (Female 20%)': 19.80747540078325,
             'Air Intake': 293.0179441912101,
             'S Ducts': 168.10360482827986,
             'Main Fuel Tank': 176.06761359034232,
             'Main Fuel Tank 2': 37.9999999999998,
             'Main Fuel Tank 3': 34.499999999999964,
             'ANAPG-81 Radar': 12.08875797898923,
             'Avioincs': 19.24999999999991,
             'F100-PW-229 (Engine)': 1.6906932461086557e-09,
             'External Casing': 169.94706630604392,
             'Nozzle Exit': 36.29436686181577,
             'Front Tire': 9.304062396477628,
             'Front Struct': 4.370104231946206,
             'Diagonal Struct': 4.370104231946232,
             'Front Tire (Stored)': 9.304062396477631,
             'Front Struct (Stored)': 4.370104231946223,
             'Diagonal Struct (Stored)': 4.

Control Surface Areas [ft^2]


{'Main_Wing,Flaps': 62.76937788659769,
 'Main_Wing,Ailerons': 16.75003751699472,
 'Main_Wing,Slats': 64.48397674038719,
 'VStab,Rudder': 25.030219000993913}

### [OBSOLETE] Parse CompGeom output txt file to find control surface areas
!! ENSURE CORRECT COMPGEOM FILE INPUT !!

In [7]:
%%capture
def cntrl_sfs_areas(compgeom_txt):
    with open(compgeom_txt, 'r') as file: 
        lines = file.readlines()
    
    for i, line in enumerate(lines):
        if "SS_Theo_Area" in line:
            start_idx = i
            break

    ss_area_df = pd.read_csv(compgeom_txt, skiprows=start_idx, sep='\s+', on_bad_lines='skip')
    display(ss_area_df)

    ss_areas = {
        "Wing_Flap_Area": ss_area_df.at[0, "SS_Theo_Area"],
        "Wing_Aileron_Area": ss_area_df.at[1, "SS_Theo_Area"],
        "Wing_Slat_Area": ss_area_df.at[2, "SS_Theo_Area"],
        "VStab_Rudder_Area": ss_area_df.at[3, "SS_Theo_Area"]
    }

    display(ss_areas)

    return ss_areas


compgeom_file = f"{vsp_filename}_CompGeom.txt"
ss_areas = cntrl_sfs_areas(compgeom_file)

### [OBSOLETE]Parse CompGeom output txt file to find wing and tail volumes
!! ENSURE CORRECT COMPGEOM FILE INPUT !!

In [8]:
%%capture
def sfs_vol(compgeom_txt):
    with open(compgeom_txt, 'r') as file: 
        lines = file.readlines()
    
    for i, line in enumerate(lines):
        if "Main_Wing" in line:
            start_idx = i
            break

    s_vol_df = pd.read_csv(compgeom_txt, skiprows=5, sep='\s+', on_bad_lines='skip', 
                           usecols=['Theo_Area', 'Wet_Area', 'Theo_Vol', 'Wet_Vol', 'Name'])
    #display(s_vol_df)

    des_name = ['Main_Wing', 'Main_Wing', 'HStab', 'VStab', 'VStab']
    s_filt = s_vol_df[s_vol_df['Name'].isin(des_name)]
    print(s_filt)

    s_vols = {
        "Wing_Vol_tot": s_filt.loc[s_filt['Name'] == 'Main_Wing', 'Theo_Vol'].astype(float).sum(),
        "Hstab_Vol_tot": s_filt.loc[s_filt['Name'] == 'HStab', 'Theo_Vol'].astype(float).iloc[0],
        "VStab_Vol_tot": s_filt.loc[s_filt['Name'] == 'VStab', 'Theo_Vol'].astype(float).sum()
    }

    display(s_vols)

    return s_vols


compgeom_file = f"{vsp_filename}_CompGeom.txt"
surf_vols = sfs_vol(compgeom_file)

### Weights - Wing
Raymer Eq. 15.1 gives statistical equations to estimate the weight of the wing. 
$\begin{equation}
W_{wing} = 0.0103K_{dw}K_{vs}(W_{dg}N_z)^{0.5}S_{w}^{0.622}A^{0.785}(t/c)_{root}^{-0.4}\times (1+\lambda)^{0.05}(\text{cos}\Lambda)^{-1.0}S_{csw}^{0.04}
\end{equation}$

Where:
- $K_{dw} = 1.0$
- $K_{vs} = 1.0$
- $W_{dg}$: Flight Design Gross Weight (typically 50-60% internal fuel)
- $N_z = 7 * 1.5 = 10.5$
- $S_w$: Trapezoidal Wing Area (incl. fuse area)
- $A$: Aspect Ratio
- $(t/c)_{root}$: Airfoil thickness at root
- $\lambda$: Taper Ratio
- $\Lambda$: Wing Sweep at 25% MAC
- $S_{csw}$: Control surface area

In [9]:
def wing_weight(geom_def, wing_weight_parm):
    #Read JSON file
    with open(f"{geom_def}", "r") as file:
        geom = json.load(file)

    wing = geom["wing"]
    hstab = geom["hstab"]
    vstab = geom["vstab"]

    #Define Parameters
    W_dg = wing_weight_parm["W_dg"]
    N_z = wing_weight_parm["N_z"]
    S_w = float(wing["S_w"])
    A = float(wing["ar_w"])
    tc_rt = wing_weight_parm["tc_rt"]
    lam_w = float(wing["lamb_w"])
    Lam_25mac = np.deg2rad(wing_weight_parm["Lambda_25"])
    '''S_csw = float(ss_areas["Wing_Aileron_Area"]) + float(ss_areas["Wing_Slat_Area"]) + float(ss_areas["Wing_Flap_Area"])
    print(f"Cntrl SFS Area: {S_csw}")'''

    S_csw2 = subsurf_areas["Main_Wing,Ailerons"] + subsurf_areas["Main_Wing,Flaps"] + subsurf_areas["Main_Wing,Slats"]
    print(f"SubSurf Area: {S_csw2}")

    wing_weight_coeff = {
        "W_dg": W_dg,
        "N_z": N_z,
        "S_w": S_w,
        "A_w": A,
        "tc_rt": tc_rt,
        "lam_w": lam_w,
        "Lam_25mac_rad": Lam_25mac,
        "S_csw": S_csw2
    }

    display(wing_weight_coeff)

    W_w = 0.0103 * (W_dg * N_z)**(0.5) * S_w**(0.622) * A**(0.785) * tc_rt**(-0.4) * (1 + lam_w)**(0.05) * (np.cos(Lam_25mac))**(-1.0) * S_csw2**(0.04)
    print(f"Wing Weight is: {W_w:.2f} lbs")

    #Conversion from lbs to slug
    M_w_slug = W_w / 32.174
    print(f"Wing Mass is: {M_w_slug:.2f} slugs")

    return M_w_slug, W_w
    

wing_weight_parm = {
    "W_dg": 50213,  #Design Gross Weight (raymer says 50% fuel, but this is full fuel)
    "N_z": 10.5,    #Design Ultimate Load Factor 
    "tc_rt": 0.06,  #Airfoil Thickness Percentage at Root
    "Lambda_25": 32, #Wing sweep at 35% MAC !! RECALCULATE THIS BASED ON COMPGEOM !!
}

geom_def = "airplane_geom.json"

M_w_slug, W_w_lbs = wing_weight(wing_weight_parm=wing_weight_parm, geom_def=geom_def)

SubSurf Area: 144.0033921439796


{'W_dg': 50213,
 'N_z': 10.5,
 'S_w': 465.0,
 'A_w': 3.5,
 'tc_rt': 0.06,
 'lam_w': 0.27,
 'Lam_25mac_rad': np.float64(0.5585053606381855),
 'S_csw': 144.0033921439796}

Wing Weight is: 4091.98 lbs
Wing Mass is: 127.18 slugs


### Weights - HStab
Raymer Eq. 15.2 statistically estimates hstab weight:
$\begin{equation}
W_{HT}=3.316\bigg( 1 + \frac{F_w}{B_h} \bigg)^{-2.0}\bigg( \frac{W_{dg}N_z}{1000} \bigg)^{0.260}S_{ht}^{0.806}
\end{equation}$

Where: 
- $F_w$: Fuselage width at horizontal tail intersection (8.3ft)
- $B_h$: Horiz. tail span
- $W_{dg}$: Flight Design Gross Weight (typically 50-60% internal fuel)
- $N_z = 10.5$
- $S_{ht}$: Horiz. tail area

In [10]:
def hstab_weight(hstab_weight_parm, geom_def):
    #Read JSON file
    with open(f"{geom_def}", "r") as file:
        geom = json.load(file)

    wing = geom["wing"]
    hstab = geom["hstab"]
    vstab = geom["vstab"]

    #Define Parameters    
    F_w = hstab_weight_parm["F_w"]
    B_h = hstab["b_HT"]
    W_dg = hstab_weight_parm["W_dg"]
    N_z = hstab_weight_parm["N_z"]
    S_ht = hstab["S_HT"]

    hstab_weight_coeffs = {
        "F_w": F_w,
        "B_h": B_h,
        "W_dg": W_dg,
        "N_z": N_z,
        "S_ht": S_ht
    }
    display(hstab_weight_coeffs)

    W_HT = 3.316 * (1 + F_w / B_h)**(-2.0) * ((W_dg * N_z) / 1000)**(0.260) * S_ht**(0.806)
    print(f"Hstab Weight is: {W_HT:.2f} lbs")

    #Conversion from lbs to slug
    M_HT_slug = W_HT / 32.174
    print(f"Hstab Mass is: {M_HT_slug:.2f} slugs")

    return M_HT_slug, W_HT



hstab_weight_parm = {
    "F_w": 7,
    "N_z": 10.5,
    "W_dg": 50213
}

M_HT_slug, W_HT_lbs = hstab_weight(hstab_weight_parm=hstab_weight_parm, geom_def=geom_def)

{'F_w': 7,
 'B_h': 17.188676378511385,
 'W_dg': 50213,
 'N_z': 10.5,
 'S_ht': 123.10441485216471}

Hstab Weight is: 413.39 lbs
Hstab Mass is: 12.85 slugs


### Weights - VStab
Raymer Eq. 15.3 statistically estimates hstab weight:
$\begin{equation}
W_{VT} = 0.452K_{rht}(1 + H_t/H_v)^{0.5}(W_{dg}N_z)^{0.488}S_{vt}^{0.718}M^{0.341}\times L_{t}^{-1.0}(1+S_r/S_{vt})^{0.348}A_{vt}^{0.233} \times (1+\lambda)^{0.25}(\text{cos}\Lambda_{vt})^{-0.323}
\end{equation}$

Where: 
- $K_{rht} = 1.0$
- $H_t/H_v = 0$
- $W_{dg}$: Flight Design Gross Weight (typically 50-60% internal fuel)
- $N_z = 10.5$
- $S_{vt}$: Vertical Tail Area
- $M$: Design max mach number
- $L_t$: Tail length; wing quarter-MAC to tail quarter-MAC
- $S_r$: Rudder Area
- $A_{vt}$: Aspect Ratio, vertical tail
- $\lambda$: Taper Ratio, vertical tail
- $\Lambda_{vt}$: Sweep at 25% MAC, vertical tail

In [16]:
def vstab_weight(geom_def, vstab_weight_parm):
    #Read JSON file
    with open(f"{geom_def}", "r") as file:
        geom = json.load(file)

    wing = geom["wing"]
    hstab = geom["hstab"]
    vstab = geom["vstab"]

    #Define Parameters
    W_dg = vstab_weight_parm["W_dg"]
    N_z = vstab_weight_parm["N_z"]
    S_vt = vstab["S_VT"]
    M = vstab_weight_parm["M"]
    L_t = vstab["L_VT"]
    S_r2 = subsurf_areas["VStab,Rudder"]
    A_VT = vstab["AR_VT"]
    lam_VT = vstab["lam_VT"]
    Lam_25mac = np.deg2rad(vstab_weight_parm["Lam_mac25"])

    vstab_weight_coeffs = {
        "W_dg": W_dg,
        "N_z": N_z,
        "S_vt": S_vt,
        "M": M,
        "L_t": L_t,
        "S_r": S_r2,
        "A_VT": A_VT,
        "lam_VT": lam_VT,
        "Lam_25mac_rad": Lam_25mac
    }
    display(vstab_weight_coeffs)

    W_VT = 0.452 * (1 + 0)**(0.5) * (W_dg * N_z)**(0.488) * S_vt**(0.718) * M**(0.341) * L_t**(-1) * (1 + S_r2 / S_vt)**(0.348) * A_VT**(0.223) * (1 + lam_VT)**(0.25) * (np.cos(Lam_25mac))**(-0.323)
    print(f"Vertical Stabilizer Weight is: {W_VT:.2f} lbf")

    #Conversion from lbs to slug
    M_VT_slug = W_VT / 32.174
    print(f"Vstab Mass is: {M_VT_slug:.2f} slugs")

    return M_VT_slug, W_VT


vstab_weight_parm = {
    "W_dg": 50213,
    "N_z": 10.5,
    "M": 1.6,
    "Lam_mac25": 35
}

M_VT_slug, W_VT_lbs = vstab_weight(geom_def=geom_def, vstab_weight_parm=vstab_weight_parm)

{'W_dg': 50213,
 'N_z': 10.5,
 'S_vt': 96.82148521893268,
 'M': 1.6,
 'L_t': 15.5,
 'S_r': 25.030219000993913,
 'A_VT': 1.35,
 'lam_VT': 0.4,
 'Lam_25mac_rad': np.float64(0.6108652381980153)}

Vertical Stabilizer Weight is: 760.34 lbf
Vstab Mass is: 23.63 slugs


### Weights - Fuel 
The fuel used is JP-5, with the desnity taken from Raymer tbl. 10.5, at 100F:
- $\rho_{\text{JP}-5}=6.65\ [\text{lb/gal}]$

It is presumed that the gallon is the U.S. Gallon, and the conversion factor to ft is: 
- $1\ \text{U.S. Gallon} = 0.1336805556 \text{ft}^3$

Then, given the volume (in ft^3) of the fuel tanks, we can calculate the amount in gallons the tank will hold: 
- $V_\text{gal} = V_{\text{ft}^3} / 0.1336805556$

Then, the weight of the volume of the fuel will be:
- $\text{W}_{\text{fuel}} = V_\text{gal} * \rho_{\text{JP}-5}$

In [12]:
def weight_fuel(desn_fuel):
    Vol_tank_gal = comp_vols["Wing Fuel Tank"] / 0.1336805556
    print(f"Volume of Wing Fuel Tanks (Integral): {Vol_tank_gal:.3f} gal")
    W_fuel_lb = Vol_tank_gal * desn_fuel
    print(f"Weight of Fuel in Fuel Tanks: {W_fuel_lb:.2f} lbs")

    M_fuel_slug = W_fuel_lb / 32.174
    print(f"Mass of Fuel in Fuel Tanks: {M_fuel_slug:.2f} slugs")

    return M_fuel_slug, W_fuel_lb

M_fuel_slug, W_fuel_lbs = weight_fuel(6.65)

Volume of Wing Fuel Tanks (Integral): 578.261 gal
Weight of Fuel in Fuel Tanks: 3845.44 lbs
Mass of Fuel in Fuel Tanks: 119.52 slugs


### Assign Wing, HStab, VStab, Fuel Tank weights

In [13]:
def assign_mass(plane, M_w, M_HT, M_VT, W_w_lbs, W_HT_lbs, W_VT_lbs, M_fuel_slug, W_fuel_lbs):
    tot_surf_mass = M_w + M_HT + M_VT
    print(f"Total Flying Surfaces Mass: {tot_surf_mass:.2f} slugs")
    print(f"Total Flying Surfaces Weight: {(tot_surf_mass * 32.174):.2f} lbf")
    print(f"Mass of Flying Surfaces + Fuel: {(tot_surf_mass + M_fuel_slug):.2f} slugs")

    #Tabulate Data
    Weights = [W_w_lbs, W_HT_lbs, W_VT_lbs, W_fuel_lbs, (tot_surf_mass + M_fuel_slug)*32.174]
    Masses = [M_w, M_HT_slug, M_VT_slug, M_fuel_slug, tot_surf_mass + M_fuel_slug]
    mlabels = ["Wing", "HStab", "VStab", "Wing Fuel Tanks", "Total"]
    mcolums = ["Component", "Weight [lbf]", "Mass [slugs]"]

    mass_df = pd.DataFrame({
        "Weight [lbf]": Weights,
        "Mass [slugs]": Masses
    }, index=mlabels)

    display(mass_df.round(3))

    #Clear and load model
    vsp.ClearVSPModel()
    vsp.ReadVSPFile(plane)

    wing_id = vsp.FindGeomsWithName("Main_Wing")[0]
    hstab_id = vsp.FindGeomsWithName("HStab")[0]
    vstab_id = vsp.FindGeomsWithName("VStab")[0]
    ftank_id = vsp.FindGeomsWithName("Wing Fuel Tank")[0]

    #Assign Density of Main Wing
    vsp.SetParmVal(wing_id, "Density", "Mass_Props", 0)
    vsp.SetParmVal(wing_id, "Shell_Flag", "Mass_Props", 1)
    vsp.SetParmVal(wing_id, "Mass_Area", "Mass_Props", M_w / comp_area["Main_Wing"])

    #Assign Density of Hstab
    vsp.SetParmVal(hstab_id, "Density", "Mass_Props", M_HT / comp_vols["HStab"])
    vsp.SetParmVal(hstab_id, "Shell_Flag", "Mass_Props", 0.0)

    #Assign Density of Vstab
    vsp.SetParmVal(vstab_id, "Density", "Mass_Props", M_VT / comp_vols["VStab"])
    vsp.SetParmVal(vstab_id, "Shell_Flag", "Mass_Props", 0.0)

    #Assign Density of Fuel Tanks
    vsp.SetParmVal(ftank_id, "Density", "Mass_Props", M_fuel_slug / comp_vols["Wing Fuel Tank"])
    vsp.SetParmVal(ftank_id, "Shell_Flag", "Mass_Props", 0.0)
    vsp.SetParmVal(ftank_id, "Mass_Prior", "Mass_Props", 1.0)

    vsp.Update()
    vsp.WriteVSPFile(plane)

    
assign_mass(plane=f"{vsp_filename}.vsp3", M_w=M_w_slug, M_HT=M_HT_slug, M_VT=M_VT_slug, W_w_lbs=W_w_lbs, W_HT_lbs=W_HT_lbs, W_VT_lbs=W_VT_lbs, M_fuel_slug=M_fuel_slug, W_fuel_lbs=W_fuel_lbs)

Total Flying Surfaces Mass: 163.66 slugs
Total Flying Surfaces Weight: 5265.71 lbf
Mass of Flying Surfaces + Fuel: 283.18 slugs


,Weight [lbf],Mass [slugs]
Wing,4091.977,127.183
HStab,413.387,12.848
VStab,760.342,23.632
Wing Fuel Tanks,3845.436,119.520
Total,9111.142,283.183


### Set up VSPAERO for NP Steady case

In [14]:
def initialize_vspaero(plane):
    vsp.ClearVSPModel()
    vsp.ReadVSPFile(plane)
 
    aero_id = vsp.FindContainer("VSPAEROSettings", 0)
    
    #Toggle Steady Analysis
    vsp.SetParmVal(aero_id, "UnsteadyType", "VSPAERO", 1.0)
    
    #Take Ref Area from Geom
    vsp.SetParmVal(aero_id, "RefFlag", "VSPAERO", 1.0)
    
    #Use 16 CPUs
    vsp.SetParmVal(aero_id, "NCPU", "VSPAERO", 16.0)
    
    #Init. CG Calc
    vsp.SetParmVal(aero_id, "NumMassSlice", "VSPAERO", 100.0)
    
    #Present Analysis to Set_3
    set_index = vsp.GetSetIndex("Set_3")
    vsp.SetParmVal(aero_id, "ThinGeomSet", "VSPAERO", float(set_index))
    
    #No VLM 
    vsp.SetParmVal(aero_id, "GeomSet", "VSPAERO", float(vsp.SET_NONE))

    vsp.Update()
    vsp.WriteVSPFile(plane)
    
    print(f"VSPAERO Settings initialized. Thin set mapped to index {set_index}.")

initialize_vspaero(plane=f"{vsp_filename}.vsp3")

VSPAERO Settings initialized. Thin set mapped to index 6.
